# Advanced document indexing

# Splitting and ingesting HTML content

## Splitting and ingesting the content of a single URL (on Cornwall)

In [2]:
# Run this cell in Google Colab before running the rest of the notebook.
%pip install -q langchain==1.0.3 langchain-openai==1.0.1 langchain-community==0.4.1 langchain-chroma==1.0.0 langchain-openrouter chromadb==1.3.0 lxml==5.4.0 html2text==2025.4.15 lark==1.2.2

### Preparing the Chroma DB collections

In [3]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import os
import getpass

try:
    from google.colab import userdata
except ImportError:
    userdata = None

OPENROUTER_API_KEY = None
if userdata is not None:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass("Enter your OPENROUTER_API_KEY: ")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ.setdefault("USER_AGENT", "building-llm-applications/ch08-colab")

embedding_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

In [4]:
corwnall_granular_collection = Chroma( #A
    collection_name="cornwall_granular",
    embedding_function=embedding_model,
)

corwnall_granular_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists

In [5]:
corwnall_coarse_collection = Chroma( #A
    collection_name="cornwall_coarse",
    embedding_function=embedding_model,
)

corwnall_coarse_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists

### Loading the HTML content with the AsyncHtmlLoader

In [6]:
from langchain_community.document_loaders import AsyncHtmlLoader

In [27]:
# destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
destination_url = "https://en.wikivoyage.org/api/rest_v1/page/html/Cornwall"

In [28]:
# html_loader = AsyncHtmlLoader(destination_url)


In [29]:
# docs = html_loader.load()

In [30]:
import requests
from urllib.parse import quote
from langchain_core.documents import Document

page_title = "Cornwall"

url = f"https://en.wikivoyage.org/api/rest_v1/page/html/{quote(page_title)}"

headers = {
    "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
    "Accept-Encoding": "gzip",
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

docs = [
    Document(
        page_content=response.text,
        metadata={"source": url, "title": page_title},
    )
]

In [31]:
len(docs)

1

### Splitting into granular chunks with the HTMLSectionSplitter

In [32]:
from langchain_text_splitters import HTMLSectionSplitter

In [33]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

In [34]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks)

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

In [35]:
granular_chunks = split_docs_into_granular_chunks(docs)

#### Ingesting granular chunks

In [36]:
corwnall_granular_collection.add_documents(documents=granular_chunks)

['db0f1afa-1154-4348-96fd-88012f567eba',
 '921acd80-e2f8-40c6-a927-1b66f4051e17',
 '4e44ff26-991f-4f30-aeca-afc0e748eff9',
 '2be4bc67-d451-4eba-a093-a6ab1710799a',
 'f682d902-c753-4f00-b80e-87a30baade7d',
 '129df43a-ce75-4b77-bcad-bee641726866',
 '29fa389c-df7a-4e49-9153-d97ed7e51753',
 '52a3427c-bde1-4afa-aad1-eae1e49e86d4',
 'bf669471-3aa7-4bd0-ae22-92ea64d771bc',
 '51aaebad-66f6-417c-9fb6-c1619c46d251',
 'c2eead94-45e6-4bdb-b35d-5034fc772f97',
 '041b0b85-df4a-408a-bd79-0385d44e77c7',
 '9b3b881a-327b-43b6-a706-090e6612c29f',
 '67887f0b-0a69-4246-ab6b-fa9e3549fd06',
 '0c0f4383-a704-43b5-9d9e-a4674aeb7f02']

#### Searching granular chunks

In [37]:
results = corwnall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='Festivals 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance of associated rituals. Some towns have a street-parade during this peri

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter

In [38]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [39]:
html2text_transformer = Html2TextTransformer()

In [40]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [41]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

In [42]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

#### Ingesting coarse chunks

In [43]:
corwnall_coarse_collection.add_documents(documents=coarse_chunks)

['4972726f-d87f-477c-ba72-74f751d7c63c',
 '624a4dba-e547-457c-b516-03dcfc40ac8d',
 '798e21f4-47d1-48ad-a468-8d215cbd08ff',
 '877b4e2e-a977-467f-9a54-ebf9a2554125',
 '156ea237-d411-47ef-949d-2e96552edd0a',
 '4ec66380-805b-45cf-b8a3-156eea65e181',
 'cf14a274-3de9-418d-a57f-f3db3da10ca2',
 'bffed8ef-bff6-4d7f-a495-d4aa074bdc3f',
 'c5e3569d-df2a-4644-a212-38a089b64781',
 'c07096ee-d484-4a60-a128-8e80c89d82d3',
 '75aeaa20-84bc-4e1d-a9f7-99370934e269',
 '7db9cc88-aaae-4c0e-851c-3bb5effc7e42',
 '3208a724-a039-42d9-843e-6e74f2079eed']

#### Searching coarse chunks

In [44]:
results = corwnall_coarse_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='_See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Cornish fishing communities. The celeb

## Splitting and ingesting the content of various URLs (across UK destinations)

### Preparing the Chroma DB collections

In [ ]:
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

uk_granular_collection.reset_collection() #B

In [ ]:
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter

In [ ]:
# Reduce this list if you want to save on processing fees
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [ ]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #C
    docs =  html_loader.load() #D

    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists
#C Loader for one destination
#D Documents of one destination

#### Searching

In [ ]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in granular_results:
    print(doc)

In [ ]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in coarse_results:
    print(doc)

In [ ]:
granular_results = uk_granular_collection.similarity_search(
    query="Beaches in Conrwall",k=4)
for doc in granular_results:
    print(doc)

In [ ]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Beaches in Cornwall",k=4)
for doc in coarse_results:
    print(doc)

# Embedding strategy

## Embedding child chunks with ParentDocumentRetriever

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Setting up the Parent Document retriever

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 16.05it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 17.39it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 18.45it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 17.61it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 12.08it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [ ]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['4000bc4c-11ad-41eb-9751-7bcd6f72654d',
 '1b4727c7-ba4c-4fc0-a395-1bb52edf23fd',
 'b23594c5-66ae-4f24-b307-383635b09858',
 '5ebe455b-cb09-41fa-9892-732a36dec28a',
 '508c5f9c-0537-4fd3-9068-327f91e6213f',
 '98f789bc-901a-4565-b07e-f56d78993423',
 'a0e75fb7-bda2-4b2a-967f-fdc73257dcc3',
 '248523cf-a3bf-47b6-80ba-234747eaf19c',
 '32d8b2d8-e59d-4470-8c86-5e60cf145520',
 '73ff6050-7428-4d47-aa50-32f1afd7f1a0',
 'f4903ff5-d516-4e27-b827-42126ad48c77',
 'cfbc6fb8-eb67-49f1-8d4a-5c6fa6021757',
 '29b94a31-6e3f-4fa1-a692-7f6e9c53f031',
 'a79e6645-1589-4f03-a13a-fa5b66cd51ff',
 '95bc1eec-0037-40ff-8faa-f69f53ea0a5b',
 '00e38278-88bf-419c-97cb-fd1e5f81297a',
 '8274d332-7265-4d32-aaf2-971f4bcea977',
 'abab2c45-e487-488a-aa23-00e6c385c64b',
 'a5c173c9-52cc-4f6e-a64a-90f50ebd2c36',
 'd989afef-0acd-471b-9b82-46f0dc390f8f',
 '7d2eb85a-6d79-4d55-b94b-795d433ad09c',
 '13d462c2-446c-4bdb-9bb0-adde90a3acce',
 'cfe67351-55df-4b97-8e14-f77f096868d8',
 'ecf78ce8-8a98-4ec0-a4da-1b5e862477b9',
 '91e78cac-c8c7-

### Performing a search on granular information

In [ ]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies. The **Cornwall All Day ticket** allows unlimited\ntravel for a calendar day. As of 2023, fares are £5 for adults and £4 for\nunder-19s. Payment is by cash or contactless. The two main bus companies are:\n\n  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).\n  * **Kernow** (part of First Bus) covers western and central Cornwall.\n\nBuses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\nse

### Comparing with direct semantic search on child chunks

In [ ]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [ ]:
len(child_docs_only)

4

In [ ]:
child_docs_only[0]

Document(id='e6356a11-7ca1-44cf-ba42-eeb9553ab838', metadata={'title': 'South Cornwall – Travel guide at Wikivoyage', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'language': 'en', 'doc_id': 'cfe67351-55df-4b97-8e14-f77f096868d8'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.')

In [ ]:
# IMPORTANT: as you can see a granular search would have identified the chunk, but it would have lost the usefulcontext about travelling in Cornwall

## Embedding child chunks with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(
        openai_api_key=OPENAI_API_KEY),
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]) #F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id #G

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_granular_chunks) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into parent coarse chunks
#E Iterate over the parent coarse chunks
#F Create child granular chunks form each parent coarse chunk
#G Link each child granular chunk to its parent coarse chunk
#H Ingest the child granular chunks into the vector store
#I Ingest the parent coarse chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 12.74it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.46it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 16.68it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.20it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.45it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke(
    "Cornwall Ranger")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="Cornwall boasts many attractions for the traveller, many lying outside of\ncities and towns amidst the Cornish landscape:\n\n  * Within the 208 m² of the Bodmin Moor, is **King Arthur's Hall** , a megalithic monument and **Brown Willy** , the highest point in Cornwall at 417 m (1,368 ft). **Dozmary Pool** is a small beautiful lake where, according to legend, King Arthur was entrusted with the sword Excalibur. There is also a reputed **Beast of the Moor** , a large wild-cat that haunts and stalks at night, but is similar in fantasy to the Loch Ness Monster, in that no one can prove it exists, though sightings, theories and track-marks abound.\n  * The **Eden Project** , near St Austell, a fabulous collection of flora from all over the planet housed in two 'space age' transparent domes.\n  * The **Lost Gardens of Heligan** \\- near Mev

In [ ]:
##IMPORTANT: same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks

### Comparing with direct semantic search on child chunks

In [ ]:
child_docs_only =  child_chunks_collection.similarity_search(
    "Cornwall Ranger")

In [ ]:
len(child_docs_only)

4

In [ ]:
child_docs_only[0]

Document(id='a18b9a25-f88f-433b-8819-2d80d8a39fcd', metadata={'doc_id': 'f69a9d0a-6153-4c74-9f19-83942aeb3876', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'language': 'en', 'title': 'South Cornwall – Travel guide at Wikivoyage'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.')

In [ ]:
## IMPORTANT: Same as before

## Embedding summaries with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

### Setting up the Multi vector retriever (similar to when embedding child chunks)

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A

summaries_collection = Chroma( #B
    collection_name="uk_summaries",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

summaries_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the summarization chain

In [ ]:
llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)

In [ ]:
summarization_chain = (
    {"document": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}") #B
    | llm
    | StrOutputParser())

#A Grab the text content from the document
#B Instantiate a prompt asking to generate summary of the provided text
#C Send the LLM the instantiated prompt
#D Extract the summary text from the response

### Ingesting the coarse chunks and related summaries into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        summary_text =  summarization_chain.invoke(
            coarse_chunk) #F
        summary_doc = Document(page_content=summary_text,
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc) #G

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a summary for the coarse chunk thorugh the summarization chain
#G Link each summary to its related coarse chunk
#H Ingest the summaries into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 11.12it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 18.13it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 18.90it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 17.62it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 11.98it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [ ]:
# COMMENT: the code above is similar to when ingesting child chunks, but it is slower because of the summarization step
# which invokes the LLM.
# The processing can be speeded up by parallelizing the outer for loop on the destination urls.

### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="### By car\n\n[edit]\n\nCornwall can be accessed by road via the A30 which runs from the end of the M5\nat Exeter, all the way through the heart of Devon and Cornwall down to Land's\nEnd. It is a grade-separated expressway as far as Carland Cross near Truro\n(the expressway is expected to be open as far as Camborne (between Redruth and\nHayle) by March 2024). You can also get to Cornwall via the A38, crossing the\nRiver Tamar at Plymouth via the Tamar Bridge, which levies a toll on eastbound\nvehicles. On summer Saturdays and during bank holiday weekends roads to\nCornwall are usually busy.\n\n### By plane\n\n[edit]\n\n50.440833-4.9952781 Cornwall Airport (**NQY** IATA) in Newquay is the main\nairport for the county, with year-round flights only from Aberdeen, Alicante,\nDublin, London Gatwick, and Manchester. During the

### Comparing with direct semantic search on summaries

In [ ]:
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

In [ ]:
len(summary_docs_only)

4

In [ ]:
summary_docs_only

[Document(id='9553aa2d-456e-4a38-8f95-28f072f13992', metadata={'doc_id': '14554480-abf6-4b35-ae7d-8580a27ecc9a'}, page_content="Cornwall offers a diverse array of attractions spanning natural beauty, legends, gardens, historic sites, arts, and heritage, including both independent sites and National Trust properties.\n\n- Natural and legendary sights: King Arthur's Hall and Brown Willy on Bodmin Moor; Dozmary Pool and tales of the Beast of the Moor.\n- Gardens and nature: The Eden Project’s two glass domes; the Lost Gardens of Heligan near Mevagissey.\n- Castles, archaeology, and coastal culture: Tintagel Castle (Arthurian legends and early medieval finds); Minack Theatre (clifftop outdoor theatre and museum); St Michael's Mount.\n- Arts and museums: Tate St Ives (modern art); National Maritime Museum, Falmouth (small-boat collection and other exhibits).\n- Mining and industrial heritage: Historic tin/copper mine sites such as Geevor Tin Mine, Poldark Mine, King Edward Mine, Crown Hill 

In [ ]:
# COMMENT: a direct search on summaries retrieves denser information, but it is missing out on useful details.
# However, you might consider using the summaries directly if after testing they prove adequate.

## Embedding hypothetical questions with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

### Setting up the Multi vector retriever (same as when embedding summaries)

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

hypothetical_questions_collection = Chroma( #B
    collection_name="uk_hypothetical_questions",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

hypothetical_questions_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the chain to generate hypothetical questions

In [ ]:
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""

    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

In [ ]:
llm_with_structured_output = ChatOpenAI(
    model="gpt-5-nano",
    openai_api_key=OPENAI_API_KEY).with_structured_output(
        HypotheticalQuestions
)

In [ ]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template( #B
        "Generate a list of exactly 4 hypothetical questions that the below text could be used to answer:\n\n{document_text}"
    )
    | llm_with_structured_output #C
    | (lambda x: x.questions) #D
)

#A Grab the text content from the document
#B Instantiate a prompt asking to generate 4 hypothetical questions on the provided text
#C Invoke the LLM configured to return an object containing the questions as a typed list of strings
#D Grab the list of questions from the response

### Ingesting the coarse chunks and related hypothetical questions into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_hypothetical_questions = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        hypothetical_questions = hypothetical_questions_chain.invoke(
            coarse_chunk) #F
        hypothetical_questions_docs = [Document(
            page_content=question, metadata={doc_key: coarse_chunk_id})
                    for question
                    in hypothetical_questions] #G

        all_hypothetical_questions.extend(hypothetical_questions_docs)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_hypothetical_questions) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a list of hypothetical questions for the coarse chunk thorugh the question generation chain
#G Link each hypothetical question to its related coarse chunk
#H Ingest the hypothetical questions into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.29it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 12.01it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  2.23it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.02it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.97it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke(
    "How can you go to Brighton from London?")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Brighton', 'title': 'Brighton – Travel guide at Wikivoyage', 'language': 'en'}, page_content='Brighton  \n---  \nClimate chart (explanation)  \n| J| F| M| A| M| J| J| A| S| O| N| D  \n---|---|---|---|---|---|---|---|---|---|---|---  \n88 8 3 |  60 8 3 |  51 9 4 |  58 12 6 |  56 16 9 |  50 18 12 |  54 20 14 |  62 21 14 |  67 18 12 |  105 15 9 |  103 11 6 |  97 9 4  \nAverage max. and min. temperatures in °C  \nPrecipitation+Snow totals in mm  \nSource: Wikipedia. Visit the Met Office for a five day forecast.  \n| Imperial conversion  \n---  \nJ| F| M| A| M| J| J| A| S| O| N| D  \n3.5 46 37 |  2.4 46 37 |  2 48 39 |  2.3 54 43 |  2.2 61 48 |  2 64 54 |  2.1 68 57 |  2.4 70 57 |  2.6 64 54 |  4.1 59 48 |  4.1 52 43 |  3.8 48 39  \nAverage max. and min. temperatures in °F  \nPrecipitation+Snow totals in inches  \n  \nThe city is close to London, and is increasingly popular with media and music\ntypes who don\'t want to live in t

### Inspecting possible questions matching our question through semantic search

In [ ]:
hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search(
    "How can you go to Brighton from London?")

In [ ]:
len(hypothetical_question_docs_only)

4

In [ ]:
hypothetical_question_docs_only

[Document(id='399bb54e-88e8-4eb1-b572-d363e8f23006', metadata={'doc_id': 'a1f51d6c-2d27-4a6c-b1a0-4f9918dc1545'}, page_content='How can you travel to Brighton by train from London, and what are the two main railway stations in the city?'),
 Document(id='65d2f229-e7e5-4516-ac09-ea93fd3feb1d', metadata={'doc_id': '53e1031c-3d92-4e7e-9398-1770b215dad1'}, page_content='What transportation options are described for getting to Brighton and for getting around the city?'),
 Document(id='fb27db61-b09a-4280-9473-cfe82759e708', metadata={'doc_id': 'f4b555cf-afe0-43e2-b10a-f59fdab3e2e2'}, page_content='What is the fastest way to travel from Gatwick to Brighton, and how long does it take by train according to the text?'),
 Document(id='320ca9aa-f328-4c06-9d18-00ef76577884', metadata={'doc_id': '79b9f77f-21ab-4ab4-bbf4-4a04d5576cbc'}, page_content='If I want to travel around Brighton all day on buses with one fare, what ticket would I buy and how much would it cost?')]

# Granular chunk expansion with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [ ]:
granular_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #A

granular_chunks_collection = Chroma( #B
    collection_name="uk_granular_chunks",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

granular_chunks_collection.reset_collection() #C

expanded_chunk_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)
#A Splitter to generate granular chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host expanded chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Ingesting granular and expanded chunks into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs) #D

    expanded_chunk_store_items = []
    for i, granular_chunk in enumerate(
        granular_chunks): #E

        this_chunk_num = i #F
        previous_chunk_num = i-1 #F
        next_chunk_num = i+1 #F

        if i==0: #F
            previous_chunk_num = None
        elif i==(len(granular_chunks)-1): #F
            next_chunk_num = None

        expanded_chunk_text = "" #G
        if previous_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                previous_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_text += granular_chunks[
            this_chunk_num].page_content #G
        expanded_chunk_text += "\n"

        if next_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                next_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_id = str(uuid.uuid4()) #H
        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text) #I

        expanded_chunk_store_item = (expanded_chunk_id,
                                     expanded_chunk_doc)
        expanded_chunk_store_items.append(
            expanded_chunk_store_item)

        granular_chunk.metadata[
            doc_key] = expanded_chunk_id #J

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        granular_chunks) #K
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items) #L

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into granular chunks
#E Iterate over the granular chunks
#F determine the index of the current chunk and its previous and next chunks
#G Assemble the text of the expanded chunk by including the previous and next chunk
#H Generate the ID of the expanded chunk
#I Create the expanded chunk document
#J Link each granular chunk to its related expanded chunk
#K Ingest the granular chunks into the vector store
#L Ingest the expanded chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.39it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 12.60it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.08it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 13.77it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 13.99it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs[0]

Document(metadata={}, page_content="Buses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\nserving a number of other towns on branch lines. For train times and fares\nvisit National Rail Enquiries.\nThe **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.\n\nThe **Lo

### Comparing with direct semantic search on granular chunks

In [ ]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [ ]:
len(child_docs_only)

4

In [ ]:
child_docs_only[0]

Document(id='a18b9a25-f88f-433b-8819-2d80d8a39fcd', metadata={'language': 'en', 'doc_id': 'f69a9d0a-6153-4c74-9f19-83942aeb3876', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.')

In [ ]:
# COMMENT: the expanded chunk has more useful context